# QLoRA adapter — citation and abstention tuning

The untuned 1.5B baseline is committed in `results/answers_baseline.md`: format 0.093,
grounded answer 0.0, 38/43 forced abstentions. This notebook trains a QLoRA adapter on
synthetic, extractive examples from the **four SFT pages only** and re-runs the same
43-question test split.

- Public repo, no token.
- The three held-out test pages never appear in training contexts.
- The adapter is saved locally and checkpointed to Drive; it is not pushed to the Hub.


In [ ]:
REPO_URL = "https://github.com/titidintsako-bit/transformers-docs-copilot.git"
EPOCHS = 2
MAX_SEQ_LENGTH = 2560
LIMIT = 0              # debug: cap train/val examples; 0 uses the full SFT split
print("repo:", REPO_URL)
print("epochs:", EPOCHS, "| max_seq_length:", MAX_SEQ_LENGTH, "| limit:", LIMIT)


## 1. Preflight

Checks the GPU, free disk, Python and that the public repo is reachable.


In [ ]:
import shutil
import subprocess

import torch

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb > 8, f"Only {free_gb:.1f} GB free in /content; free space and retry"

print(f"python: {__import__('sys').version.split()[0]}")
print(f"torch: {torch.__version__} | {props.name} | {props.total_memory/1e9:.1f} GB VRAM")
print(f"free disk: {free_gb:.1f} GB")
head = subprocess.run(["git", "ls-remote", REPO_URL, "HEAD"], capture_output=True, text=True, check=True)
print("repo head:", head.stdout.split()[0][:12])


## 2. Get the project

Clones the public repo, or pulls the latest commit. A dirty/partial clone is re-cloned.


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/transformers-docs-copilot")
if PROJECT_DIR.exists() and not (PROJECT_DIR / "pyproject.toml").exists():
    shutil.rmtree(PROJECT_DIR)
if PROJECT_DIR.exists():
    pull = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        capture_output=True, text=True,
    )
    if pull.returncode != 0:
        print("Repo update failed; re-cloning.")
        print(pull.stderr[-400:])
        shutil.rmtree(PROJECT_DIR)
if not PROJECT_DIR.exists():
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise SystemExit("git clone failed:\n" + clone.stderr[-1200:])
os.chdir(PROJECT_DIR)
print("project:", Path.cwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


## 3. Install the training runtime

Adds PEFT to the pinned inference stack. It does not replace Colab's PyTorch.


In [ ]:
import subprocess
import sys

packages = [
    "PyYAML==6.0.2",
    "transformers==5.24.0",
    "accelerate==1.16.0",
    "safetensors==0.8.0",
    "huggingface-hub==2.4.0",
    "bitsandbytes==0.50.2",
    "peft==0.22.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import bitsandbytes
import peft
import transformers

print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("bitsandbytes", bitsandbytes.__version__)


## 4. Build the SFT split

Synthetic extractive targets from the four SFT pages, plus 55 unrelated abstention
examples. The held-out pages are excluded from the training retriever itself.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "src.fetch_docs", "--check"], check=True)
subprocess.run([sys.executable, "-m", "src.index"], check=True)
subprocess.run([sys.executable, "-m", "src.build_sft"], check=True)

manifest = json.loads(Path("data/sft/manifest.json").read_text())
print(json.dumps(manifest, indent=2))


## 5. Train the adapter

Only assistant answer tokens are supervised; prompt tokens are masked. On a T4 this is
roughly 10–20 minutes for the full split.


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.train_qlora",
    "--train", "data/sft/train.jsonl",
    "--val", "data/sft/val.jsonl",
    "--output-dir", "artifacts/adapter",
    "--epochs", str(EPOCHS),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 6. Evaluate on the same test split

`behavior_accuracy` still requires the model itself to choose answer vs abstain. A forced
abstention is a failure even though the delivered output is safe.


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, "-m", "src.eval_answers",
    "--split", "test",
    "--model", "base",
    "--quantize", "on",
    "--adapter", "artifacts/adapter",
    "--out-prefix", "results/answers_tuned",
    "--labels", "results/label_sheet_tuned.csv",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from pathlib import Path
from IPython.display import Markdown, display

payload = json.loads(Path("results/answers_tuned.json").read_text())
print(json.dumps(payload["overall"], indent=2))
display(Markdown(Path("results/answers_tuned.md").read_text()))


## 7. Compare with the untuned baseline


In [ ]:
import json
from pathlib import Path

base = json.loads(Path("results/answers_baseline.json").read_text())["overall"]
tuned = json.loads(Path("results/answers_tuned.json").read_text())["overall"]
keys = [
    "behavior_accuracy", "answered_rate", "grounded_answer_rate",
    "format_rate", "citation_hit_rate", "model_abstain_rate_on_negative",
    "delivered_abstention_rate_on_negative", "false_abstain_rate_on_answerable",
    "median_latency_s",
]
print(f"{'metric':42s} {'base':>8s} {'tuned':>8s}")
for key in keys:
    print(f"{key:42s} {str(base.get(key)):>8s} {str(tuned.get(key)):>8s}")


## 8. Checkpoint adapter and results to Drive (optional)


In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive not mounted; skipping checkpoint:", exc)
else:
    dest = Path("/content/drive/MyDrive/transformers-docs-copilot-results")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree("artifacts/adapter", dest / "adapter", dirs_exist_ok=True)
    for name in ("answers_tuned.md", "answers_tuned.json", "label_sheet_tuned.csv"):
        shutil.copy2(Path("results") / name, dest / name)
    print("saved adapter and tuned results to", dest)


## 9. Next steps

1. Paste the `overall` JSON and the baseline-vs-tuned table into the chat.
2. If format/abstention improved, download `label_sheet_tuned.csv` and fill the human
   support/correctness columns.
3. If format is still near zero, the 1.5B cannot learn the contract from this data;
   switch to the 3B/4B base before spending more time on prompt variants.
